# Hydra Python Benchmark Analysis

Analysis of test timing data from the Hydra Python implementation.

This notebook analyzes results from the Common Test Suite benchmarks, which can be generated with:
```bash
cd hydra-python
./bin/benchmark.sh --all -o test_timings.csv
```

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Find the benchmark CSV
HYDRA_ROOT = Path.cwd().parent.parent
CSV_PATH = HYDRA_ROOT / "hydra-python" / "test_timings.csv"

print(f"Loading benchmark data from: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} test results")

Loading benchmark data from: /Users/josh/projects/github/CategoricalData/hydra/hydra-python/test_timings.csv
Loaded 2714 test results


In [2]:
# Overview statistics
print("=" * 60)
print("OVERVIEW")
print("=" * 60)

for test_type in df['test_type'].unique():
    subset = df[df['test_type'] == test_type]
    total_ms = subset['duration_ms'].sum()
    print(f"\n{test_type.upper()} TESTS:")
    print(f"  Total tests: {len(subset)}")
    print(f"  Passed: {len(subset[subset['status'] == 'passed'])}")
    print(f"  Failed: {len(subset[subset['status'] == 'failed'])}")
    print(f"  Skipped: {len(subset[subset['status'] == 'skipped'])}")
    print(f"  Total time: {total_ms/1000:.1f}s ({total_ms/60000:.1f} min)")
    print(f"  Mean time: {subset['duration_ms'].mean():.1f}ms")
    print(f"  Median time: {subset['duration_ms'].median():.1f}ms")
    print(f"  Max time: {subset['duration_ms'].max():.1f}ms")

OVERVIEW

KERNEL TESTS:
  Total tests: 1857
  Passed: 1637
  Failed: 62
  Skipped: 158
  Total time: 164.4s (2.7 min)
  Mean time: 88.6ms
  Median time: 0.0ms
  Max time: 53910.0ms

GENERATION TESTS:
  Total tests: 857
  Passed: 857
  Failed: 0
  Skipped: 0
  Total time: 0.0s (0.0 min)
  Mean time: 0.0ms
  Median time: 0.0ms
  Max time: 10.0ms


## 1. Slowest Individual Tests

Tests sorted by duration, showing which specific test cases are most expensive.

In [3]:
# Top 30 slowest tests
slowest = df.nlargest(30, 'duration_ms')[['test_type', 'hydra_path', 'status', 'duration_ms']].copy()
slowest['duration_s'] = slowest['duration_ms'] / 1000
slowest['pct_of_total'] = (slowest['duration_ms'] / df['duration_ms'].sum() * 100).round(2)

print("Top 30 Slowest Tests")
print("=" * 100)
for i, row in slowest.iterrows():
    print(f"{row['duration_s']:8.1f}s ({row['pct_of_total']:5.2f}%) [{row['status']:7}] {row['hydra_path'][:70]}")

# Cumulative percentage of top N tests
top_10_pct = df.nlargest(10, 'duration_ms')['duration_ms'].sum() / df['duration_ms'].sum() * 100
top_20_pct = df.nlargest(20, 'duration_ms')['duration_ms'].sum() / df['duration_ms'].sum() * 100
print(f"\nTop 10 tests account for {top_10_pct:.1f}% of total time")
print(f"Top 20 tests account for {top_20_pct:.1f}% of total time")

Top 30 Slowest Tests
    53.9s (32.78%) [passed ] common/checking/Nominal types/Eliminations/Union eliminations/Polymorp
     9.5s ( 5.80%) [passed ] common/eta expansion/Let terms/partial application of a let-bound func
     5.4s ( 3.27%) [passed ] common/inference/Expected failures/Complex constraint failures/Functio
     5.1s ( 3.10%) [passed ] common/checking/Nominal types/Eliminations/Record eliminations/Multi-p
     5.0s ( 3.02%) [passed ] common/checking/Nominal types/Unions/Multi-parameter polymorphic injec
     4.8s ( 2.92%) [passed ] common/checking/Fundamentals/Let terms/Let with complex expressions/co
     4.8s ( 2.91%) [passed ] common/checking/Fundamentals/Applications/Higher-order applications/fu
     3.7s ( 2.25%) [passed ] common/inference/Expected failures/Constraint solver edge cases/Comple
     2.8s ( 1.70%) [passed ] common/inference/Algorithm W test cases/STLC to System F/#7
     2.8s ( 1.68%) [passed ] common/inference/Fundamentals/Let terms/Recursive and mutuall

## 2. Slowest Test Groups

Aggregated by test group hierarchy to identify which categories of tests are most expensive.

In [4]:
# Parse the hydra_path into hierarchy levels
def parse_path_levels(path):
    """Extract hierarchy levels from hydra_path."""
    parts = path.split('/')
    return {
        'level_1': parts[0] if len(parts) > 0 else None,  # e.g., 'common'
        'level_2': parts[1] if len(parts) > 1 else None,  # e.g., 'inference'
        'level_3': parts[2] if len(parts) > 2 else None,  # e.g., 'Fundamentals'
        'level_4': parts[3] if len(parts) > 3 else None,  # e.g., 'Let terms'
        'level_5': parts[4] if len(parts) > 4 else None,  # e.g., 'Let-polymorphism'
    }

# Add hierarchy columns
levels = df['hydra_path'].apply(parse_path_levels).apply(pd.Series)
df_with_levels = pd.concat([df, levels], axis=1)

print("Hierarchy levels extracted")
print(f"Level 1 categories: {df_with_levels['level_1'].nunique()}")
print(f"Level 2 categories: {df_with_levels['level_2'].nunique()}")
print(f"Level 3 categories: {df_with_levels['level_3'].nunique()}")

Hierarchy levels extracted
Level 1 categories: 1
Level 2 categories: 31
Level 3 categories: 260


In [5]:
# Level 2 groups (e.g., common/inference, common/checking)
level2_stats = df_with_levels.groupby(['level_1', 'level_2']).agg({
    'duration_ms': ['sum', 'mean', 'median', 'max', 'count'],
    'status': lambda x: (x == 'passed').sum()
}).round(1)
level2_stats.columns = ['total_ms', 'mean_ms', 'median_ms', 'max_ms', 'count', 'passed']
level2_stats['total_s'] = (level2_stats['total_ms'] / 1000).round(1)
level2_stats['pct'] = (level2_stats['total_ms'] / df['duration_ms'].sum() * 100).round(1)
level2_stats = level2_stats.sort_values('mean_ms', ascending=False)  # Sort by MEAN

print("\nLevel 2 Test Groups (sorted by AVERAGE time per test)")
print("=" * 100)
print(f"{'Group':<40} {'Tests':>6} {'Passed':>7} {'Mean':>10} {'Median':>10} {'Max':>10}")
print("-" * 100)
for (l1, l2), row in level2_stats.iterrows():
    group_name = f"{l1}/{l2}"
    print(f"{group_name:<40} {int(row['count']):>6} {int(row['passed']):>7} {row['mean_ms']:>8.0f}ms {row['median_ms']:>8.0f}ms {row['max_ms']:>8.0f}ms")


Level 2 Test Groups (sorted by AVERAGE time per test)
Group                                     Tests  Passed       Mean     Median        Max
----------------------------------------------------------------------------------------------------
common/checking                             335     330      311ms       10ms    53910ms
common/eta expansion                         49      49      229ms        0ms     9530ms
common/inference                            280     271      153ms       10ms     5370ms
common/hydra.lib.lists primitives           339     339       19ms        0ms     2730ms
common/hydra.lib.chars primitives            40      40        0ms        0ms       10ms
common/hydra.lib.math primitives            264     264        0ms        0ms       20ms
common/rewriting                            145     145        0ms        0ms       10ms
common/reduction                            120      65        0ms        0ms        0ms
common/monads                              

In [6]:
# Level 3 groups - sorted by AVERAGE time per test (which groups have slow tests?)
level3_stats = df_with_levels.groupby(['level_1', 'level_2', 'level_3']).agg({
    'duration_ms': ['sum', 'mean', 'median', 'max', 'count']
}).round(1)
level3_stats.columns = ['total_ms', 'mean_ms', 'median_ms', 'max_ms', 'count']
level3_stats['total_s'] = (level3_stats['total_ms'] / 1000).round(1)
level3_stats = level3_stats.sort_values('mean_ms', ascending=False)  # Sort by MEAN

print("\nTop 20 Level 3 Test Groups (sorted by AVERAGE time per test)")
print("=" * 110)
print(f"{'Group':<55} {'Tests':>6} {'Mean':>10} {'Median':>10} {'Max':>10}")
print("-" * 110)
for (l1, l2, l3), row in level3_stats.head(20).iterrows():
    group_name = f"{l1}/{l2}/{l3}"
    print(f"{group_name:<55} {int(row['count']):>6} {row['mean_ms']:>8.0f}ms {row['median_ms']:>8.0f}ms {row['max_ms']:>8.0f}ms")


Top 20 Level 3 Test Groups (sorted by AVERAGE time per test)
Group                                                    Tests       Mean     Median        Max
--------------------------------------------------------------------------------------------------------------
common/eta expansion/Let terms                               3     3177ms        0ms     9530ms
common/checking/Nominal types                              131      612ms       10ms    53910ms
common/inference/Algorithm W test cases                     12      537ms      170ms     2790ms
common/hydra.lib.lists primitives/span                       8      379ms        0ms     2730ms
common/hydra.lib.lists primitives/dropWhile                  8      370ms        0ms     2680ms
common/checking/Fundamentals                               103      207ms       20ms     4810ms
common/eta expansion/Non-expansion of eliminations which produce functions      2      200ms      200ms      400ms
common/inference/Fundamentals           

In [7]:
# Level 4 groups - sorted by AVERAGE time per test
level4_stats = df_with_levels.groupby(['level_2', 'level_3', 'level_4']).agg({
    'duration_ms': ['sum', 'mean', 'median', 'max', 'count']
}).round(1)
level4_stats.columns = ['total_ms', 'mean_ms', 'median_ms', 'max_ms', 'count']
level4_stats['total_s'] = (level4_stats['total_ms'] / 1000).round(1)
level4_stats = level4_stats.sort_values('mean_ms', ascending=False)  # Sort by MEAN

print("\nTop 25 Level 4 Test Groups (sorted by AVERAGE time per test)")
print("=" * 120)
print(f"{'Group':<70} {'Tests':>6} {'Mean':>10} {'Median':>10} {'Max':>10}")
print("-" * 120)
for (l2, l3, l4), row in level4_stats.head(25).iterrows():
    group_name = f"{l2}/{l3}/{l4}" if l4 else f"{l2}/{l3}"
    print(f"{group_name:<70} {int(row['count']):>6} {row['mean_ms']:>8.0f}ms {row['median_ms']:>8.0f}ms {row['max_ms']:>8.0f}ms")


Top 25 Level 4 Test Groups (sorted by AVERAGE time per test)
Group                                                                   Tests       Mean     Median        Max
------------------------------------------------------------------------------------------------------------------------
eta expansion/Let terms/partial application of a let-bound function         3     3177ms        0ms     9530ms
inference/Expected failures/Complex constraint failures                     5     1418ms      570ms     5370ms
hydra.lib.lists primitives/span/span less than 3                            2     1365ms     1365ms     2730ms
hydra.lib.lists primitives/dropWhile/drop while less than 3                 2     1340ms     1340ms     2680ms
checking/Nominal types/Eliminations                                        77      950ms       30ms    53910ms
inference/Expected failures/Constraint solver edge cases                    6      757ms      130ms     3700ms
inference/Algorithm W test cases/STLC to

## 3. Kernel vs Generation Test Comparison

Compare timing between kernel tests (runtime) and generation tests (generated code execution).

In [8]:
# Check if we have both test types
test_types = df['test_type'].unique()
print(f"Test types present: {list(test_types)}")

if len(test_types) > 1:
    # Compare kernel vs generation
    comparison = df.groupby('test_type').agg({
        'duration_ms': ['sum', 'mean', 'median', 'std', 'max', 'count']
    }).round(2)
    comparison.columns = ['total_ms', 'mean_ms', 'median_ms', 'std_ms', 'max_ms', 'count']
    comparison['total_s'] = comparison['total_ms'] / 1000
    comparison['total_min'] = comparison['total_ms'] / 60000
    
    print("\nKernel vs Generation Comparison")
    print("=" * 80)
    print(comparison.to_string())
    
    # If we have matching tests, compare them directly
    # This would require tests with same hydra_path in both types
else:
    print("\nOnly one test type present. Run with --all to include generation tests.")
    print("\nTo generate both test types:")
    print("  cd hydra-python")
    print("  ./bin/benchmark.sh --all -o test_timings.csv")

Test types present: ['kernel', 'generation']

Kernel vs Generation Comparison
            total_ms  mean_ms  median_ms   std_ms   max_ms  count  total_s  total_min
test_type                                                                            
generation      10.0     0.01        0.0     0.34     10.0    857     0.01   0.000167
kernel      164440.0    88.55        0.0  1311.89  53910.0   1857   164.44   2.740667


In [9]:
# If we have generation tests, try to match them with kernel tests by hydra_path
if 'generation' in df['test_type'].values and 'kernel' in df['test_type'].values:
    # Get unique paths by aggregating (in case of duplicates)
    kernel_df = df[df['test_type'] == 'kernel'].groupby('hydra_path')['duration_ms'].first()
    gen_df = df[df['test_type'] == 'generation'].groupby('hydra_path')['duration_ms'].first()
    
    # Find matching paths
    common_paths = kernel_df.index.intersection(gen_df.index)
    
    if len(common_paths) > 0:
        print(f"\nFound {len(common_paths)} tests with matching paths in both kernel and generation")
        
        matched = pd.DataFrame({
            'kernel_ms': kernel_df.loc[common_paths],
            'generation_ms': gen_df.loc[common_paths]
        })
        matched['speedup'] = matched['kernel_ms'] / matched['generation_ms'].replace(0, 0.01)  # Avoid div by zero
        matched = matched.sort_values('kernel_ms', ascending=False)
        
        print("\nTop 20 matched tests (sorted by kernel time):")
        print("=" * 90)
        print(f"{'hydra_path':<50} {'kernel':>12} {'generation':>12} {'speedup':>10}")
        print("-" * 90)
        for path, row in matched.head(20).iterrows():
            path_short = path[:48] + '..' if len(path) > 50 else path
            speedup_str = f"{row['speedup']:.0f}x" if row['generation_ms'] > 0 else "∞"
            print(f"{path_short:<50} {row['kernel_ms']:>10.0f}ms {row['generation_ms']:>10.0f}ms {speedup_str:>10}")
        
        # Summary statistics
        print("\n\nSummary of matched tests:")
        print(f"  Total kernel time: {matched['kernel_ms'].sum()/1000:.1f}s")
        print(f"  Total generation time: {matched['generation_ms'].sum()/1000:.3f}s")
        print(f"  Mean kernel time: {matched['kernel_ms'].mean():.1f}ms")
        print(f"  Mean generation time: {matched['generation_ms'].mean():.3f}ms")
        
        # Group by test category
        matched['category'] = [p.split('/')[1] if '/' in p else 'unknown' for p in matched.index]
        cat_summary = matched.groupby('category').agg({
            'kernel_ms': ['sum', 'mean', 'count'],
            'generation_ms': ['sum', 'mean']
        }).round(1)
        cat_summary.columns = ['kernel_total', 'kernel_mean', 'count', 'gen_total', 'gen_mean']
        cat_summary = cat_summary.sort_values('kernel_total', ascending=False)
        
        print("\n\nMatched tests by category:")
        print(f"{'Category':<35} {'Count':>6} {'Kernel Total':>12} {'Gen Total':>12}")
        print("-" * 70)
        for cat, row in cat_summary.iterrows():
            print(f"{cat:<35} {int(row['count']):>6} {row['kernel_total']/1000:>10.1f}s {row['gen_total']/1000:>10.3f}s")
    else:
        print("\nNo matching hydra_paths found between kernel and generation tests.")
        print("Run the benchmark with updated path mapping to enable comparison.")


Found 702 tests with matching paths in both kernel and generation

Top 20 matched tests (sorted by kernel time):
hydra_path                                               kernel   generation    speedup
------------------------------------------------------------------------------------------
common/hydra.lib.lists primitives/span/span less..       2730ms          0ms          ∞
common/hydra.lib.lists primitives/dropWhile/drop..       2680ms          0ms          ∞
common/hydra.lib.lists primitives/dropWhile/drop..        150ms          0ms          ∞
common/hydra.lib.lists primitives/span/span no e..        150ms          0ms          ∞
common/hydra.lib.lists primitives/span/span all ..        150ms          0ms          ∞
common/hydra.lib.lists primitives/dropWhile/drop..        130ms          0ms          ∞
common/hydra.lib.lists primitives/sortOn/sort by..        100ms          0ms          ∞
common/hydra.lib.lists primitives/sortOn/sort by..        100ms          0ms          ∞
com

## 4. Test Duration Distribution

Understanding the distribution of test durations helps identify outliers and typical performance.

In [10]:
# Duration distribution statistics
print("Duration Distribution (passed tests only)")
print("=" * 60)

passed = df[df['status'] == 'passed']['duration_ms']

percentiles = [50, 75, 90, 95, 99, 99.9]
print(f"\nPercentiles:")
for p in percentiles:
    val = np.percentile(passed, p)
    print(f"  {p:5.1f}%: {val:10.1f}ms ({val/1000:.2f}s)")

# Bucket tests by duration
buckets = [
    (0, 10, "< 10ms (very fast)"),
    (10, 100, "10-100ms (fast)"),
    (100, 1000, "100ms-1s (moderate)"),
    (1000, 10000, "1-10s (slow)"),
    (10000, 60000, "10-60s (very slow)"),
    (60000, float('inf'), "> 60s (extremely slow)")
]

print(f"\nDuration Buckets:")
for low, high, label in buckets:
    count = len(passed[(passed >= low) & (passed < high)])
    pct = count / len(passed) * 100
    print(f"  {label:<25}: {count:5} tests ({pct:5.1f}%)")

Duration Distribution (passed tests only)

Percentiles:
   50.0%:        0.0ms (0.00s)
   75.0%:        0.0ms (0.00s)
   90.0%:       20.0ms (0.02s)
   95.0%:      100.0ms (0.10s)
   99.0%:      900.7ms (0.90s)
   99.9%:     5236.9ms (5.24s)

Duration Buckets:
  < 10ms (very fast)       :  2028 tests ( 81.3%)
  10-100ms (fast)          :   340 tests ( 13.6%)
  100ms-1s (moderate)      :   103 tests (  4.1%)
  1-10s (slow)             :    22 tests (  0.9%)
  10-60s (very slow)       :     1 tests (  0.0%)
  > 60s (extremely slow)   :     0 tests (  0.0%)


In [11]:
# Where does the time actually go?
print("\nTime Distribution by Duration Bucket")
print("=" * 60)
print("(Shows what fraction of total test time is spent in each bucket)")

total_time = passed.sum()
for low, high, label in buckets:
    bucket_time = passed[(passed >= low) & (passed < high)].sum()
    pct = bucket_time / total_time * 100
    print(f"  {label:<25}: {bucket_time/1000:8.1f}s ({pct:5.1f}% of time)")


Time Distribution by Duration Bucket
(Shows what fraction of total test time is spent in each bucket)
  < 10ms (very fast)       :      0.0s (  0.0% of time)
  10-100ms (fast)          :      8.1s (  4.9% of time)
  100ms-1s (moderate)      :     38.0s ( 23.1% of time)
  1-10s (slow)             :     64.3s ( 39.1% of time)
  10-60s (very slow)       :     53.9s ( 32.8% of time)
  > 60s (extremely slow)   :      0.0s (  0.0% of time)


## 5. Failed and Skipped Tests Analysis

In [12]:
# Analyze failed tests
failed = df[df['status'] == 'failed']
print(f"Failed Tests: {len(failed)}")
print("=" * 80)

if len(failed) > 0:
    # Group failures by level 2
    failed_with_levels = pd.concat([failed, failed['hydra_path'].apply(parse_path_levels).apply(pd.Series)], axis=1)
    failure_by_group = failed_with_levels.groupby(['level_2', 'level_3']).size().sort_values(ascending=False)
    
    print("\nFailures by test group:")
    for (l2, l3), count in failure_by_group.head(15).items():
        print(f"  {count:3} failures in {l2}/{l3}")
    
    print(f"\n\nSample failed test paths:")
    for path in failed['hydra_path'].head(10):
        print(f"  {path}")

Failed Tests: 62

Failures by test group:
   18 failures in annotations/arbitrary annotations
   13 failures in annotations/descriptions
    6 failures in annotations/layered annotations
    4 failures in hydra.lib.flows primitives/pure
    2 failures in monads/map
    2 failures in monads/bind
    2 failures in hydra.lib.flows primitives/mapMaybe
    2 failures in monads/pure
    2 failures in hydra.lib.flows primitives/map
    2 failures in hydra.lib.flows primitives/bind
    1 failures in hydra.lib.flows primitives/mapElems
    1 failures in hydra.lib.flows primitives/mapList
    1 failures in hydra.lib.flows primitives/foldl
    1 failures in hydra.lib.flows primitives/mapSet
    1 failures in hydra.lib.flows primitives/fail


Sample failed test paths:
  common/annotations/arbitrary annotations/unset one of multiple annotations #1
  common/annotations/arbitrary annotations/set multiple annotations #2
  common/annotations/arbitrary annotations/unset one of multiple annotations #2
  

In [13]:
# Analyze skipped tests
skipped = df[df['status'] == 'skipped']
print(f"\nSkipped Tests: {len(skipped)}")
print("=" * 80)

if len(skipped) > 0:
    skipped_with_levels = pd.concat([skipped, skipped['hydra_path'].apply(parse_path_levels).apply(pd.Series)], axis=1)
    skipped_by_group = skipped_with_levels.groupby(['level_2', 'level_3']).size().sort_values(ascending=False)
    
    print("\nSkipped tests by group:")
    for (l2, l3), count in skipped_by_group.head(15).items():
        print(f"  {count:3} skipped in {l2}/{l3}")


Skipped Tests: 158

Skipped tests by group:
   32 skipped in reduction/hoistCaseStatements
   23 skipped in reduction/hoistSubterms
   12 skipped in JSON parsing/primitives
   10 skipped in JSON serialization/primitives
   10 skipped in JSON coder/literal types
    9 skipped in JSON parsing/strings
    9 skipped in JSON serialization/strings
    6 skipped in inference/Fundamentals
    6 skipped in JSON parsing/whitespace handling
    5 skipped in JSON parsing/arrays
    5 skipped in JSON serialization/arrays
    4 skipped in checking/Nominal types
    4 skipped in JSON serialization/nested structures
    4 skipped in JSON serialization/objects
    4 skipped in JSON parsing/objects


## 6. Performance Patterns

Identify patterns in what makes tests slow.

In [14]:
# Look for patterns in slow test names
slow_tests = df[df['duration_ms'] > 5000]['hydra_path'].tolist()

# Count keyword occurrences in slow tests
keywords = ['polymorphic', 'recursive', 'mutual', 'nested', 'let', 'lambda', 
            'fold', 'map', 'either', 'maybe', 'flow', 'inference', 'checking']

print("Keyword frequency in tests > 5 seconds")
print("=" * 50)
keyword_counts = {}
for kw in keywords:
    count = sum(1 for path in slow_tests if kw.lower() in path.lower())
    if count > 0:
        keyword_counts[kw] = count

for kw, count in sorted(keyword_counts.items(), key=lambda x: -x[1]):
    pct = count / len(slow_tests) * 100
    print(f"  {kw:<15}: {count:3} tests ({pct:5.1f}%)")

print(f"\nTotal slow tests (>5s): {len(slow_tests)}")

Keyword frequency in tests > 5 seconds
  polymorphic    :   2 tests ( 50.0%)
  checking       :   2 tests ( 50.0%)
  let            :   1 tests ( 25.0%)
  fold           :   1 tests ( 25.0%)
  inference      :   1 tests ( 25.0%)

Total slow tests (>5s): 4


In [15]:
# Compare inference vs checking performance
inference = df_with_levels[df_with_levels['level_2'] == 'inference']
checking = df_with_levels[df_with_levels['level_2'] == 'checking']

print("Inference vs Checking Comparison")
print("=" * 60)
print(f"\nInference tests:")
print(f"  Count: {len(inference)}")
print(f"  Total time: {inference['duration_ms'].sum()/1000:.1f}s")
print(f"  Mean: {inference['duration_ms'].mean():.1f}ms")
print(f"  Median: {inference['duration_ms'].median():.1f}ms")
print(f"  Max: {inference['duration_ms'].max():.1f}ms")

print(f"\nChecking tests:")
print(f"  Count: {len(checking)}")
print(f"  Total time: {checking['duration_ms'].sum()/1000:.1f}s")
print(f"  Mean: {checking['duration_ms'].mean():.1f}ms")
print(f"  Median: {checking['duration_ms'].median():.1f}ms")
print(f"  Max: {checking['duration_ms'].max():.1f}ms")

Inference vs Checking Comparison

Inference tests:
  Count: 280
  Total time: 42.8s
  Mean: 152.7ms
  Median: 10.0ms
  Max: 5370.0ms

Checking tests:
  Count: 335
  Total time: 104.1s
  Mean: 310.8ms
  Median: 10.0ms
  Max: 53910.0ms


## 7. Export Summary Tables

In [16]:
# Export summary tables for cross-implementation comparison
output_dir = Path.cwd()

# Level 2 summary
level2_export = level2_stats.reset_index()
level2_export['group'] = level2_export['level_1'] + '/' + level2_export['level_2']
level2_export = level2_export[['group', 'count', 'passed', 'total_s', 'mean_ms', 'max_ms', 'pct']]
level2_export.to_csv(output_dir / 'python_level2_summary.csv', index=False)
print(f"Exported: {output_dir / 'python_level2_summary.csv'}")

# Top 100 slowest tests
slowest_export = df.nlargest(100, 'duration_ms')[['test_type', 'hydra_path', 'status', 'duration_ms']]
slowest_export.to_csv(output_dir / 'python_slowest_100.csv', index=False)
print(f"Exported: {output_dir / 'python_slowest_100.csv'}")

print("\nThese files can be used for cross-implementation comparison.")

Exported: /Users/josh/projects/github/CategoricalData/hydra/analysis/python/python_level2_summary.csv
Exported: /Users/josh/projects/github/CategoricalData/hydra/analysis/python/python_slowest_100.csv

These files can be used for cross-implementation comparison.
